# Roboflow shoe-brand inference

Klasifikasi gambar menjadi **adidas**, **converse**, atau **nike** menggunakan model hosted `shoe-classification-k9ajr/2`.

In [ ]:
import json
import os
from pathlib import Path
from urllib.parse import urlparse

from inference_sdk import InferenceConfiguration, InferenceHTTPClient

API_URL = "https://serverless.roboflow.com"
MODEL_ID = "shoe-classification-k9ajr/2"

Atur `IMAGE_SOURCE` ke path file lokal atau URL gambar publik. API key harus tersedia sebagai environment variable `ROBOFLOW_API_KEY`.

In [ ]:
IMAGE_SOURCE = "YOUR_IMAGE.jpg"
OUTPUT_PATH = Path("../outputs/prediction.json")

In [ ]:
def is_http_url(value: str) -> bool:
    parsed_url = urlparse(value)
    return parsed_url.scheme in {"http", "https"} and bool(parsed_url.netloc)


def validate_image_source(image_source: str) -> str:
    if is_http_url(image_source):
        return image_source

    image_path = Path(image_source).expanduser().resolve()
    if not image_path.is_file():
        raise FileNotFoundError(f"Image file not found: {image_path}")
    return str(image_path)


def get_api_key() -> str:
    api_key = os.getenv("ROBOFLOW_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("ROBOFLOW_API_KEY is not set in the environment.")
    return api_key

In [ ]:
image_source = validate_image_source(IMAGE_SOURCE)
client = InferenceHTTPClient(
    api_url=API_URL,
    api_key=get_api_key(),
).configure(InferenceConfiguration(api_key_transport="header"))

result = client.infer(image_source, model_id=MODEL_ID)
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
predictions = result.get("predictions", [])
if predictions:
    best_prediction = max(predictions, key=lambda prediction: prediction.get("confidence", 0))
    print(f"Brand: {best_prediction.get('class', 'unknown')}")
    print(f"Confidence: {best_prediction.get('confidence', 0):.2%}")
else:
    print("No predictions returned.")

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Prediction saved to: {OUTPUT_PATH.resolve()}")